In [11]:
import pandas as pd
import re

input_log_path = "/Users/fredikamionka/PycharmProjects/Projects_for_CV/Files_Project_for_CV/logs.txt"
output_parquet_path = "/Users/fredikamionka/PycharmProjects/Projects_for_CV/Files_Project_for_CV/logs_clean.parquet"

log_pattern = re.compile(
    r'(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) '
    r'(?P<level>INFO|WARN|ERROR) '
    r'(?P<message>.*) '
    r'user_id=(?P<user_id>\d+) '
    r'ip=(?P<ip>\d+\.\d+\.\d+\.\d+)'
)

parsed_records = []

with open(input_log_path, "r") as log_file:

    for line in log_file:

        line = line.strip()
        match = log_pattern.match(line)

        if match:

            parsed_record = match.groupdict()
            parsed_record["status"] = "ok"

        else:

            parsed_record = {
                "timestamp": None,
                "level": None,
                "message": None,
                "user_id": None,
                "ip": None,
                "status": "error",
                "raw_line": line
            }

        parsed_records.append(parsed_record)

parsed_logs_df = pd.DataFrame(parsed_records)

parsed_logs_df["timestamp"] = pd.to_datetime(parsed_logs_df["timestamp"], errors="coerce")
parsed_logs_df["user_id"] = pd.to_numeric(parsed_logs_df["user_id"], errors="coerce")

parsed_logs_df.to_parquet(output_parquet_path)

print(parsed_logs_df.head())
print(parsed_logs_df["status"].value_counts())

            timestamp  level           message  user_id             ip status  \
0 2026-04-09 08:00:00   WARN       User logout     79.0   192.168.1.22     ok   
1 2026-04-09 08:00:04   INFO        User login     53.0  192.168.1.203     ok   
2 2026-04-09 08:00:08  ERROR  Password changed     32.0   192.168.1.61     ok   
3 2026-04-09 08:00:09  ERROR       User logout    117.0  192.168.1.188     ok   
4 2026-04-09 08:00:16  ERROR        User login    105.0   192.168.1.98     ok   

  raw_line  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4      NaN  
status
error    882
ok       114
Name: count, dtype: int64


In [8]:
print(parsed_logs_df[parsed_logs_df["status"] == "error"]["raw_line"].head(20))

33    08:02:45 ERROR File uploaded user_id=120 ip=19...
34    08:01:08 ERROR File deleted user_id=108 ip=192...
35    08:02:55 INFO File uploaded user_id=103 ip=192...
36    08:03:00 WARN User login user_id=63 ip=192.168...
37    ERROR Settings updated user_id=57 ip=192.168.1...
38    WARN User login user_id=31 ip=192.168.1.243 20...
39    File deleted user_id=76 ip=192.168.1.3 2026-04...
40    uploaded user_id=106 ip=192.168.1.230 2026-04-...
41    login user_id=28 ip=192.168.1.197 2026-04-09 0...
42    updated user_id=43 ip=192.168.1.201 2026-04-09...
43    logout user_id=100 ip=192.168.1.172 2026-04-09...
44    user_id=97 ip=192.168.1.103 2026-04-09 08:01:3...
45    user_id=49 ip=192.168.1.80 2026-04-09 08:03:50...
46    user_id=14 ip=192.168.1.48 2026-04-09 08:02:21...
47    user_id=56 ip=192.168.1.86 ### SYSTEM GLITCH #...
48    INFO Settings updated user_id=9 ip=192.168.1.8...
49    Password changed user_id=104 ip=192.168.1.175 ...
50    File uploaded user_id=58 ip=192.168.1.222 